In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_Sri_Aurobindo_Marg_Delhi_DPCC_2023.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,251.0,164.0,159.0,97.0,75.0,55.0,42.0,53.0,86.0,130.0,342.0,359.0
1,2,352.0,180.0,179.0,63.0,64.0,100.0,47.0,69.0,84.0,123.0,346.0,339.0
2,3,403.0,175.0,118.0,186.0,80.0,89.0,74.0,63.0,81.0,116.0,471.0,291.0
3,4,347.0,198.0,99.0,NaN,236.0,167.0,115.0,68.0,80.0,148.0,390.0,308.0
4,5,343.0,NaN,109.0,98.0,177.0,174.0,62.0,53.0,72.0,151.0,434.0,284.0
5,6,421.0,185.0,109.0,99.0,196.0,208.0,46.0,72.0,61.0,165.0,408.0,254.0
6,7,371.0,261.0,151.0,95.0,219.0,251.0,207.0,77.0,48.0,143.0,378.0,270.0
7,8,374.0,103.0,173.0,125.0,104.0,154.0,44.0,84.0,43.0,122.0,392.0,294.0
8,9,444.0,185.0,92.0,125.0,202.0,96.0,NaN,93.0,37.0,129.0,416.0,283.0
9,10,429.0,186.0,129.0,124.0,181.0,94.0,NaN,99.0,40.0,NaN,239.0,301.0


In [4]:
df.info()
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        40 non-null     object 
 1   January    36 non-null     float64
 2   February   32 non-null     float64
 3   March      35 non-null     float64
 4   April      33 non-null     float64
 5   May        34 non-null     float64
 6   June       35 non-null     float64
 7   July       25 non-null     float64
 8   August     35 non-null     float64
 9   September  34 non-null     float64
 10  October    36 non-null     float64
 11  November   35 non-null     float64
 12  December   35 non-null     float64
dtypes: float64(12), object(1)
memory usage: 4.3+ KB


(41, 13)

In [5]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [6]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [7]:
# Convert all columns except 'Day' to numeric values
for col in df.columns:
    if col != 'Day':
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Fill missing values with column mean
df_filled = df.fillna(df.mean(numeric_only=True))

In [8]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [9]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready.head()

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,251.0,164.00000,159.0,97.000000,75.0,55.0,42.00,53.0,86.0,130.0,342.0,359.0
1,2,352.0,180.00000,179.0,63.000000,64.0,100.0,47.00,69.0,84.0,123.0,346.0,339.0
2,3,403.0,175.00000,118.0,186.000000,80.0,89.0,52.04,63.0,81.0,116.0,471.0,291.0
3,4,347.0,198.00000,99.0,126.606061,236.0,88.4,52.04,68.0,80.0,148.0,390.0,308.0
4,5,343.0,178.40625,109.0,98.000000,177.0,88.4,52.04,53.0,72.0,151.0,434.0,284.0
